# Session Matching And Automated Recommendation Tool (SMART)
This is the processing for AIM 2026

# Setup

## Import Statements

In [1]:
import importlib
import session_organizer
import pandas as pd
from google import genai
# Only needed if you want to reload the module after making changes
importlib.reload(session_organizer)


<module 'session_organizer' from '/Users/josephdvorak/Library/CloudStorage/OneDrive-UniversityofKentucky/Programming/AI/SMART/Session-Creation-Package/session_organizer.py'>

## Step 1: Load and Examine Data

In [2]:
# First, examine the Excel file structure
file_path = "Submissions_852925_all.xlsx"
df_temp = pd.read_excel(file_path)

print("Available columns:")
for i, col in enumerate(df_temp.columns):
    print(f"{i}: {col}")

print(f"\nFile contains {len(df_temp)} rows and {len(df_temp.columns)} columns")
print("\nFirst few rows preview:")
print(df_temp.head())

Available columns:
0: Submission ID
1: Submission Created Date & Time
2: External reference
3: Submission Completed Date & Time
4: Submission Status
5: Acceptance Status
6: # Reviews
7: Rating
8: Std Dev
9: Owner-E-mail Address
10: Owner-First Name
11: Owner-Last Name
12: Owner-Company/University
13: Owner-City
14: Owner-State
15: Owner-Country
16: Owner-CC Email
17: Owner-What is your ASABE membership number?  If you are not an ASABE member, please skip.
18: Owner-Address One
19: Owner-Address Two
20: Owner-Phone
21: Owner-Company/University.1
22: Owner-Title
23: Owner-Default Language
24: Owner-Zip
25: Owner-Biography
26: Owner-Profile Photo
27: Owner-Status
28: Owner-Submission Limit
29: Owner-Test Profile
30: Submission-Call for Abstracts-Submission ID - 7 digits
31: Submission-Call for Abstracts-CHAIRS-enter your notes to other organizers or yourself here for session movement
32: Submission-Call for Abstracts-Select Your Session Preference
33: Submission-Call for Abstracts-Anticip

In [3]:
# Define your column selections based on the output above
TITLE_COLUMN = 'Submission-Call for Abstracts-Presentation Title-Character max 160'  
ABSTRACT_COLUMN = 'Submission-Call for Abstracts-Abstract-Character max 4000-Abstracts will only be used to group into topical sessions and evaluate quality of talk.'  
ID_COLUMN = 'Submission-Call for Abstracts-Submission ID - 7 digits'  



## Step 2: Load Embedding Model

# Process Steps

## No Hybrid Sessions Example

### Load Data

In [4]:
# Load the data using the session_organizer function
df, title_column, abstract_column, abstract_id_column, topic_column = session_organizer.load_presentations(
    file_path, 
    Title_name=TITLE_COLUMN,
    Abstract_name=ABSTRACT_COLUMN,
    Abstract_ID_name=ID_COLUMN
)

print(f"Loaded {len(df)} presentations successfully")
print(f"Title column: {title_column}")
print(f"Abstract column: {abstract_column}")
print(f"ID column: {abstract_id_column}")
print(f"Topic column: {topic_column}")

Loaded 1592 presentations successfully
Title column: Title
Abstract column: Abstract
ID column: Abstract ID
Topic column: Title and Abstract


### Perform the Embedding

## Google Gemini Embedding Model
First test with a smaller sub sample.
### Embedding test run

In [5]:
df_sample = df.head(3)

In [6]:
# Load environment variables
from dotenv import load_dotenv
import os
api_key = None  # Replace with your API key string if you want to provide it directly
load_dotenv(".env")
# Check for API key in environment variables if not provided
if api_key is None:
    if "GEMINI_API_KEY" not in os.environ:
        raise ValueError(
            "API key must be provided or in environmental variables. GEMINI_API_KEY not found in environment variables. Please set it in your .env file."
        )
    else:
        api_key = os.environ["GEMINI_API_KEY"]

# Validate API key
if not api_key:
    raise ValueError("API key is required to use Google GenAI.")

In [7]:
client = genai.Client()

# Create a copy to avoid SettingWithCopyWarning
df_sample = df_sample.copy()
# Initialize the embedding column
df_sample['embedding'] = None

for idx, row in df_sample.iterrows():
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=[row[topic_column]]
    )
    embedding = response.embeddings[0].values
    print(f"Presentation ID: {row[abstract_id_column]}")
    print(f"Embedding: {embedding}\n")
    df_sample.at[idx, 'embedding'] = embedding

Presentation ID: 2600239.0
Embedding: [-0.010482273, 0.011415574, 0.030194858, -0.05439659, 0.010134398, -0.010770797, -0.007089493, 0.013247777, -0.00471532, 0.010137753, -0.0045366483, -0.00939094, 0.0069848727, 0.028363587, 0.108260944, 0.02273609, 0.003095531, 0.01584786, 0.017954342, -0.006390234, 0.007913525, -0.018247152, -0.004446714, 0.005015684, -0.003488794, 0.0068012867, 0.014964477, 0.029403744, 0.03426977, 0.0057316427, 0.0073116273, 0.015142689, 0.0120093785, 0.016521376, 0.0116748605, 0.009838545, 0.028856251, -0.0027965424, 0.007048727, 0.012612102, -0.004775025, 0.0077705598, -0.013824561, 0.0079158135, 0.027256612, 0.010923704, 0.012660041, -0.026458098, 0.022936124, 0.00905112, -0.01987354, -0.010748793, -0.034180112, -0.14105412, -0.017228086, -0.003560801, -0.010630973, -0.0050890767, 0.016509784, 0.0068053156, -0.007296098, 0.017347127, -0.023588933, -0.028264582, 0.015306959, -0.014817535, 0.007433769, 0.021746043, -0.02905694, -0.016529636, 0.012154125, -0.0045

### Run all presentations through the embedding model

In [8]:
client = genai.Client()
print(f"Processing {len(df)} presentations for embeddings.")
# Initialize the embedding column
df['embedding'] = None

for idx, row in df.iterrows():
    print(f"Processing presentation {idx + 1} of {len(df)} (ID: {row[abstract_id_column]})")
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=[row[topic_column]]
    )
    embedding = response.embeddings[0].values
    df.at[idx, 'embedding'] = embedding

Processing 1592 presentations for embeddings.
Processing presentation 1 of 1592 (ID: 2600239.0)
Processing presentation 2 of 1592 (ID: 2600002.0)
Processing presentation 3 of 1592 (ID: 2600110.0)
Processing presentation 4 of 1592 (ID: 2600003.0)
Processing presentation 5 of 1592 (ID: 2600004.0)
Processing presentation 6 of 1592 (ID: 2600005.0)
Processing presentation 7 of 1592 (ID: 2600006.0)
Processing presentation 8 of 1592 (ID: 2600008.0)
Processing presentation 9 of 1592 (ID: 2600007.0)
Processing presentation 10 of 1592 (ID: 2600898.0)
Processing presentation 11 of 1592 (ID: 2600785.0)
Processing presentation 12 of 1592 (ID: 2600009.0)
Processing presentation 13 of 1592 (ID: 2600046.0)
Processing presentation 14 of 1592 (ID: 2600010.0)
Processing presentation 15 of 1592 (ID: 2600504.0)
Processing presentation 16 of 1592 (ID: 2600852.0)
Processing presentation 17 of 1592 (ID: 2600880.0)
Processing presentation 18 of 1592 (ID: 2600577.0)
Processing presentation 19 of 1592 (ID: 26012

## Save dataframe with embeddings

In [9]:
df.to_parquet("presentations_with_embeddings.parquet", compression='snappy')
print("Embeddings saved to presentations_with_embeddings.parquet")

Embeddings saved to presentations_with_embeddings.parquet


### Remove Duplicates and Near-Duplicates

In [10]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
def remove_duplicates(df, similarity_func, threshold=0.95, embedding_column='embedding'):
    """
    Remove near-duplicate rows based on a similarity threshold.
    Args:
        df (pd.DataFrame): DataFrame containing presentation data with 'embedding' column.
        similarity_func (callable): Function to compute similarity between embeddings (e.g., cosine_similarity).
        threshold (float): Similarity threshold for considering items as near duplicates.
    Returns:
        pd.DataFrame: Presentations DataFrame with near-duplicate rows removed.
    """
    # Extract embeddings from the 'embedding' column into a numpy array
    embeddings_list = df[embedding_column].tolist()
    embeddings_array = np.array(embeddings_list)
    
    # Get the indices from the dataframe (important for referencing)
    presentation_indices = df.index.tolist()
    
    # Create a set to store the indices we want to REMOVE
    indices_to_remove = set()
    
    # Calculate the similarity matrix for the embeddings
    similarity_matrix = similarity_func(embeddings_array, embeddings_array)
    
    # Iterate through the upper triangle of the similarity matrix
    num_items = similarity_matrix.shape[0]

    for i in range(num_items):
        for j in range(i + 1, num_items):
            if similarity_matrix[i, j] >= threshold:
                # Get the actual DataFrame indices for positions i and j
                idx_i = presentation_indices[i]
                idx_j = presentation_indices[j]
                print(f"Near duplicate found: Index {idx_i} and Index {idx_j} (Similarity: {similarity_matrix[i, j]:.4f}).")
                # Remove the item with the LOWER index (keep the higher one)
                if idx_i < idx_j:
                    indices_to_remove.add(idx_i)
                else:
                    indices_to_remove.add(idx_j)

    # Convert the set of indices to remove into a list
    indices_to_remove_list = sorted(list(indices_to_remove))

    print(f"\nFound {len(indices_to_remove_list)} near-duplicate presentations to remove (keeping highest index).")
    print(f"Indices to remove: {indices_to_remove_list}")

    # --- Perform the removal ---
    df_cleaned = df.drop(index=indices_to_remove_list)

    # --- Verification ---
    print(f"\nFinal number of presentations: {len(df_cleaned)}")

    # Reset the index of the DataFrame to ensure it is clean and sequential
    df_cleaned = df_cleaned.reset_index(drop=True)
    
    return df_cleaned

In [11]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
similarity_threshold = 0.99
# Remove near-duplicate presentations based on the similarity threshold
df_cleaned = remove_duplicates(df, similarity_func=cosine_similarity, threshold=similarity_threshold, embedding_column='embedding')

Near duplicate found: Index 200 and Index 201 (Similarity: 1.0000).
Near duplicate found: Index 213 and Index 214 (Similarity: 1.0000).
Near duplicate found: Index 215 and Index 216 (Similarity: 1.0000).
Near duplicate found: Index 358 and Index 359 (Similarity: 1.0000).
Near duplicate found: Index 454 and Index 1590 (Similarity: 1.0000).
Near duplicate found: Index 465 and Index 466 (Similarity: 1.0000).
Near duplicate found: Index 541 and Index 926 (Similarity: 0.9976).
Near duplicate found: Index 748 and Index 749 (Similarity: 0.9972).
Near duplicate found: Index 848 and Index 1283 (Similarity: 1.0000).
Near duplicate found: Index 904 and Index 905 (Similarity: 1.0000).
Near duplicate found: Index 904 and Index 906 (Similarity: 1.0000).
Near duplicate found: Index 905 and Index 906 (Similarity: 1.0000).
Near duplicate found: Index 1028 and Index 1029 (Similarity: 1.0000).
Near duplicate found: Index 1030 and Index 1031 (Similarity: 0.9926).
Near duplicate found: Index 1043 and Index

### Create Sessions

In [17]:
session_column_name = 'Session Code'
df_sessions, labels, metadata = session_organizer.create_sessions_w_hybrid(df, embedding_model.similarity, df_presentation_embeddings=df_presentation_embeddings,
                                                                                     max_sessions=100, min_session_size=8, tree_merge_stop=1, cluster_column_name=session_column_name)
df[session_column_name] = labels
print(f"Created {metadata['n_clusters']} sessions with {metadata['n_assigned_items']} presentations.")
print(f"Unassigned Presentations: {metadata['n_unassigned_items']}")

Created 100 sessions with 1559 presentations.
Unassigned Presentations: 0


### Analyze Sessions

- session_coherence = "Are presentations within this session similar?" (internal session quality)
- session_distinctiveness = "Is this session's topic unique compared to others?" (relative session positioning)
- presentation_session_fit = "Does this presentation match the topic of others in the session?" (presentation fit)

Session Coherence measures cluster cohesion. It reflects how tighly grouped the topic of presentations within the session are.

Session Distinctiveness measures how unique each session's topic is. High values mean the session has a clear, focused theme that's different from other sessions. Low values suggest either the session mixes different topics or overlaps too much with other sessions.

Presentation-Session Fit is an individual presentations's average similarity to other presentation in its session. Generically, it can be referred to as "within_cluster_fit", "cluster_membership_strength", or "local_cohesion_score".

In [18]:
embeddings_only = df_presentation_embeddings.drop(columns=[session_organizer.COLUMNS['EMBEDDING_MODEL']])
pres_similarities_matrix = embedding_model.similarity(embeddings_only.values, embeddings_only.values)
# Convert to numpy if needed
if hasattr(pres_similarities_matrix, 'cpu'):
    pres_similarities_matrix = pres_similarities_matrix.cpu().numpy()
elif hasattr(pres_similarities_matrix, 'numpy'):
    pres_similarities_matrix = pres_similarities_matrix.numpy()

df['presentation_session_fit'],df_sessions['session_coherence'], df_sessions['session_distinctiveness'], df_session_session_similarity  = session_organizer.calculate_placement_metrics(
    df_presentations=df,
    df_sessions=df_sessions,
    pres_similarities_matrix=pres_similarities_matrix,
    session_column_name=session_column_name
)

### Create Session Titles & Keywords

#### Ollama

In [19]:
# Test if Ollama is accessible
import requests
try:
    response = requests.get("http://localhost:11434/api/tags")
    print(f"Ollama status: {response.status_code}")
    if response.status_code == 200:
        models = response.json()['models']
        print(f"Available models: {[m['name'] for m in models]}")
    else:
        print("Ollama server not responding correctly")
except Exception as e:
    print(f"Cannot connect to Ollama: {e}")
    print("Make sure to run 'ollama serve' first")

Ollama status: 200
Available models: ['llama3.2:latest']


In [20]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='ollama:llama3.2:latest')
# Generate titles and keywords for all sessions
# Display sample results
print(df_sessions_sample.head().to_string(index=False))

Using model: llama3.2:latest
Processing session 0 (1/3)...
  ✓ Generated titles for session 0
Processing session 1 (2/3)...
  ✓ Generated titles for session 1
Processing session 2 (3/3)...
  ✓ Generated titles for session 2

Total processing time: 45.02 seconds
Average time per session: 15.01 seconds
 cluster_id  session_size                                                                               gen_presentation_indices hybrid_invited_presentations final_session_title  session_coherence  session_distinctiveness                       Ollama: llama3.2:latest Title 1                    Ollama: llama3.2:latest Title 2                           Ollama: llama3.2:latest Title 3                                                                                                                       Ollama: llama3.2:latest Keywords
          0            19 [877, 596, 778, 243, 980, 1258, 146, 1121, 1210, 1288, 439, 1110, 667, 941, 1129, 1116, 452, 133, 210]                           []     

#### Sentence Transformers

In [ ]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='llama-3.2-local')
# Generate titles and keywords for all sessions
# df_sessions = generate_session_titles_and_keywords(df_sessions, df, topic_column)

# Display sample results
print(df_sessions_sample.head().to_string(index=False))

LLaMA model loaded successfully
Processing session 0 (1/3)...


In [ ]:
if 'model' in globals() or 'model' in locals():
    del model
    # Optionally, you can try to explicitly trigger garbage collection
    # import gc
    # gc.collect()
    print("LLaMA model has been flagged for unloading. Resources will be freed by the garbage collector.")
else:
    print("Model variable 'model' not found, or already unloaded.")

#### Gemini

In [10]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='gemini-2.0-flash')
# Generate titles and keywords for all sessions
# df_sessions = generate_session_titles_and_keywords(df_sessions, df, topic_column)

# Display sample results
print(df_sessions_sample.head().to_string(index=False))

Processing session 0 (1/3)...
  ✓ Generated titles for session 0
Processing session 1 (2/3)...
  ✓ Generated titles for session 1
Processing session 2 (3/3)...
  ✓ Generated titles for session 2

Total processing time: 7.1328 seconds
Average time per session: 2.3776 seconds
 cluster_id  session_size                                                                               gen_presentation_indices hybrid_invited_presentations final_session_title  session_coherence  session_distinctiveness                             Gemini Title 1                                  Gemini Title 2                              Gemini Title 3                                                                Gemini Keywords
          0            19 [877, 596, 778, 243, 980, 1258, 146, 1121, 1210, 1288, 439, 1110, 667, 941, 1129, 1116, 452, 133, 210]                           []         Not Set Yet           0.516857                 0.031459   AI & Robotics for Precision Weed Control      Sensing and Automat

### Match Committees to Related Sessions

In [11]:
# Read the committee file from CSV/Excel with flexible column selection
committee_file_path = 'ASABE Committees.csv'  # Update this path as needed (can also use .xlsx)

# Load committees
df_committees, committee_name_column, description_column, combined_column = session_organizer.load_committees(
    committee_file_path,
    Committee_Name_column='Committee_Name',  # Actual column name in your file
    Description_column='Description',        # Actual column name in your file
    committee_name_column='Committee_Name',  # Desired output column name
    description_column='Description',        # Desired output column name
    combined_column='Name_Description'       # Combined column for embeddings
)

print(f"Loaded {len(df_committees)} committees")
print(f"Committee name column: {committee_name_column}")
print(f"Description column: {description_column}")
print(f"Combined column: {combined_column}")
print("\nFirst few committees:")
print(df_committees[[committee_name_column, description_column]].head())

# Generate embeddings for committees using the combined column
df_committee_embeddings = session_organizer.embed_documents(df_committees, combined_column, embedding_model)

Loaded 108 committees
Committee name column: Committee_Name
Description column: Description
Combined column: Name_Description

First few committees:
                                      Committee_Name  \
0  ASE-09 Environmental Quality Coordinating Comm...   
1                          ASE-12 Forest Engineering   
2  ASE-134 Fertilizers, Soil Conditioners & US TA...   
3              ASE-16 Engineering for Sustainability   
4  ASE-347 and US TAG TC 347 Data-driven agrifood...   

                                         Description  
0  Leads and coordinates the activities of ASABE ...  
1  Forested landscapes are essential for clean wa...  
2  US Technical Advisory Group for ISO TC 134. Le...  
3  ASE-16 leads and coordinates ASABE activities ...  
4  Standardization in the field of big-picture, d...  


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [12]:
# Find the most similar committees for each session
session_committee_matches = session_organizer.find_most_similar_committees_by_presentations(
    df_sessions, 
    df_presentation_embeddings, 
    df_committees, 
    df_committee_embeddings, 
    top_n=3,
)

Processing session 0 with 19 presentations...
Processing session 1 with 13 presentations...
Processing session 2 with 15 presentations...
Processing session 3 with 14 presentations...
Processing session 4 with 18 presentations...
Processing session 5 with 14 presentations...
Processing session 6 with 11 presentations...
Processing session 7 with 16 presentations...
Processing session 8 with 15 presentations...
Processing session 9 with 27 presentations...
Processing session 10 with 11 presentations...
Processing session 11 with 11 presentations...
Processing session 12 with 14 presentations...
Processing session 13 with 26 presentations...
Processing session 14 with 13 presentations...
Processing session 15 with 16 presentations...
Processing session 16 with 13 presentations...
Processing session 17 with 19 presentations...
Processing session 18 with 16 presentations...
Processing session 19 with 12 presentations...
Processing session 20 with 15 presentations...
Processing session 21 w

In [13]:
df_sessions = session_organizer.add_committee_matches_to_clusters(df_sessions, session_committee_matches)
# Display a sample of the results
print(f"\nSample of top committee matches:")

print(df_sessions.head(10).to_string(index=False))


Sample of top committee matches:
 cluster_id  session_size                                                                                                                   gen_presentation_indices hybrid_invited_presentations final_session_title  session_coherence  session_distinctiveness                             Top Committee Match                                    2nd Committee Match                        3rd Committee Match Top Committee Similarity 2nd Committee Similarity 3rd Committee Similarity
          0            19                                     [877, 596, 778, 243, 980, 1258, 146, 1121, 1210, 1288, 439, 1110, 667, 941, 1129, 1116, 452, 133, 210]                           []         Not Set Yet           0.516857                 0.031459               MS-45 Soil-Plant-Machine Dynamics                         NRES-244 Irrigation Management   PRS-702 Crop & Feed Processing & Storage                 0.430897                 0.424146                 0.422593
        

## Hybrid Sessions Example

### Load Invited Presentaions from Hybrid Sessions

If you have existing hybrid sessions with pre-assigned presentations, you can load them.

In [5]:
# First, examine the Excel file structure
file_path = "Example Hybrid Session Invited Presentations.csv"
# Determine file type and read accordingly
if file_path.lower().endswith('.csv'):
    df_temp = pd.read_csv(file_path)
elif file_path.lower().endswith(('.xlsx', '.xls')):
    df_temp = pd.read_excel(file_path)
else:
    raise ValueError(f"Unsupported file format. Please use CSV (.csv) or Excel (.xlsx, .xls) files.")

print("Available columns:")
for i, col in enumerate(df_temp.columns):
    print(f"{i}: {col}")

print(f"\nFile contains {len(df_temp)} rows and {len(df_temp.columns)} columns")
print("\nFirst few rows preview:")
print(df_temp.head())

Available columns:
0: Session
1: Title
2: Abstract
3: Submission ID - 7 digits
4: Technical Community
5: Presenter: First Name
6: Presenter: Last Name

File contains 6 rows and 7 columns

First few rows preview:
                                             Session  \
0  AI-Powered Remote Sensing for Crop and Soil He...   
1  AI-Powered Remote Sensing for Crop and Soil He...   
2  AI-Powered Remote Sensing for Crop and Soil He...   
3  Open-Source “pyfao56” Evapotranspiration and W...   
4  Open-Source “pyfao56” Evapotranspiration and W...   

                                               Title  \
0  Weed-AI: Open and standardised sharing of anno...   
1  Unified Deep Learning Framework for Crop-Weed ...   
2  Accurate Pixel-Wise Object Detection Framework...   
3  The “pyfao56” software package for Python: Cod...   
4  The pyfao56 automatic irrigation scheduling al...   

                                            Abstract  \
0  Machine vision for weed recognition is critica...   
1 

In [6]:
# Example: Load hybrid sessions from CSV/Excel file
hybrid_file_path = "Example Hybrid Session Invited Presentations.csv"  # Update this path as needed

# Load hybrid sessions using the flexible function
df_hybrid_presentations, df_hybrid_sessions, hybrid_session_col, title_col, abstract_col, abstract_id_col, topic_col = session_organizer.load_hybrid_sessions(
    hybrid_file_path,
    Session_column='Session',              # Actual column name in your file
    Title_column='Title',                  # Actual column name in your file  
    Abstract_column='Abstract',            # Actual column name in your file
    Abstract_ID_column='Submission ID - 7 digits',  # Actual column name in your file
    session_column='Session',              # Desired output column name
    title_column='Title',                  # Desired output column name
    abstract_column='Abstract',            # Desired output column name
    abstract_id_column='Abstract ID',      # Desired output column name
    topic_column='Title and Abstract'      # Combined column for embeddings
)

print(f"Loaded {len(df_hybrid_presentations)} hybrid presentations")
print(f"Session column: {hybrid_session_col}")
print(f"Title column: {title_col}")
print(f"Abstract column: {abstract_col}")
print(f"ID column: {abstract_id_col}")
print(f"Topic column: {topic_col}")

print("\nHybrid Sessions Summary:")
print(df_hybrid_sessions[[session_organizer.COLUMNS['CLUSTER_ID'], session_organizer.COLUMNS['SESSION_SIZE'], session_organizer.COLUMNS['HYBRID_SESSION_TITLE']]].to_string(index=False))

Loaded 6 hybrid presentations in 2 sessions
Session mapping: {'AI-Powered Remote Sensing for Crop and Soil Health Monitoring': 1, 'Open-Source “pyfao56” Evapotranspiration and Water Balance Tool for Water Management': 2}
Loaded 6 hybrid presentations
Session column: Session
Title column: Title
Abstract column: Abstract
ID column: Abstract ID
Topic column: Title and Abstract

Hybrid Sessions Summary:
 cluster_id  session_size                                                                 hybrid_session_title
          1             3                        AI-Powered Remote Sensing for Crop and Soil Health Monitoring
          2             3 Open-Source “pyfao56” Evapotranspiration and Water Balance Tool for Water Management


### Embed Invited Presentations from Hybrid Sessions

In [7]:
# Generate embeddings for hybrid presentations (if needed for analysis)
df_hybrid_embeddings = session_organizer.embed_documents(df_hybrid_presentations, topic_col, embedding_model)
print(f"✓ Created embeddings for hybrid presentations with shape: {df_hybrid_embeddings.shape}")
print(f"✓ Embedding model used: {df_hybrid_embeddings[session_organizer.COLUMNS['EMBEDDING_MODEL']].iloc[0]}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Created embeddings for hybrid presentations with shape: (6, 385)
✓ Embedding model used: Unknown (sentence-transformers/all-MiniLM-L6-v2)


### Load Regular Presentaitons

In [8]:
# Load the data using the session_organizer function
file_path = "1.29.25 Abstracts.xlsx"
df, title_column, abstract_column, abstract_id_column, topic_column = session_organizer.load_presentations(
    file_path, 
    Title_name=TITLE_COLUMN,
    Abstract_name=ABSTRACT_COLUMN,
    Abstract_ID_name=ID_COLUMN
)

print(f"Loaded {len(df)} presentations successfully")
print(f"Title column: {title_column}")
print(f"Abstract column: {abstract_column}")
print(f"ID column: {abstract_id_column}")
print(f"Topic column: {topic_column}")

Loaded 1601 presentations successfully
Title column: Title
Abstract column: Abstract
ID column: Abstract ID
Topic column: Title and Abstract


### Embed Regular Presentations

In [9]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', trust_remote_code=True)
print(f"Base model: {embedding_model.model_card_data.base_model}")
df_presentation_embeddings = session_organizer.embed_documents(df, topic_column, embedding_model)
print(f"Embeddings shape: {df_presentation_embeddings.shape}")
print(f"Embedding model used: {df_presentation_embeddings[session_organizer.COLUMNS['EMBEDDING_MODEL']].iloc[0]}")

Base model: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/51 [00:00<?, ?it/s]

Embeddings shape: (1601, 385)
Embedding model used: Unknown (sentence-transformers/all-MiniLM-L6-v2)


### Remove Duplicates and Near Duplicates
This only applies to the regular presentation list.

In [10]:
similarity_threshold = 0.99
# Remove near-duplicate presentations based on the similarity threshold
df, df_presentation_embeddings = session_organizer.remove_duplicates(df, df_presentation_embeddings, similarity_func=embedding_model.similarity, threshold=similarity_threshold)

Near duplicate found: Index 41 and Index 42 (Similarity: 1.0000).
Near duplicate found: Index 77 and Index 78 (Similarity: 1.0000).
Near duplicate found: Index 77 and Index 79 (Similarity: 1.0000).
Near duplicate found: Index 78 and Index 79 (Similarity: 1.0000).
Near duplicate found: Index 156 and Index 157 (Similarity: 1.0000).
Near duplicate found: Index 221 and Index 222 (Similarity: 1.0000).
Near duplicate found: Index 223 and Index 224 (Similarity: 0.9930).
Near duplicate found: Index 269 and Index 270 (Similarity: 1.0000).
Near duplicate found: Index 279 and Index 280 (Similarity: 1.0000).
Near duplicate found: Index 325 and Index 326 (Similarity: 1.0000).
Near duplicate found: Index 354 and Index 355 (Similarity: 1.0000).
Near duplicate found: Index 410 and Index 411 (Similarity: 1.0000).
Near duplicate found: Index 458 and Index 459 (Similarity: 0.9998).
Near duplicate found: Index 476 and Index 477 (Similarity: 1.0000).
Near duplicate found: Index 496 and Index 1348 (Similari

### Create Sessions with Hybrid Sessions

In [11]:
session_column_name = 'Session Code'
df_sessions, labels, metadata = session_organizer.create_sessions_w_hybrid(df, embedding_model.similarity, df_presentation_embeddings=df_presentation_embeddings,
                                                                                     df_hybrid_presentations=df_hybrid_presentations,
                                                                                     hybrid_session_column=hybrid_session_col, df_hybrid_embeddings=df_hybrid_embeddings,
                                                                                     max_sessions=100, min_session_size=8, tree_merge_stop=1, cluster_column_name=session_column_name)
df[session_column_name] = labels
print(f"Created {metadata['n_clusters']} sessions with {metadata['n_assigned_items']} presentations.")
print(f"Unassigned Presentations: {metadata['n_unassigned_items']}")
    

Created 100 sessions with 1559 presentations.
Unassigned Presentations: 0


#### Optional: Add hybrid invited presentations

Hybrid invited presentations are included in the sessions, but they are not in the presentation dataframe that is used for analyzing the sessions. The session creation algorithm does not have any control over the placement of these presentations. If the analysis is to look at the ability of the algorithm to create sessions and place presentations, they should not be included. If the goal is to see the session statistics overall, they should be included. This code will add them to end of the presentation dataframe with the correct session id so that they are considered.

In [12]:
def print_df_info(*dfs):
    local_vars = locals()
    for df in dfs:
        # Find variable name(s) pointing to this object
        var_names = [name for name, val in globals().items() if val is df]
        name_str = var_names[0] if var_names else "DataFrame"
        print(f"Available columns in {name_str}:")
        for i, col in enumerate(df.columns):
            print(f"{i}: {col}")
        print(f"\nDataframe contains {len(df)} rows and {len(df.columns)} columns\n")

# Usage:
print_df_info(df, df_hybrid_presentations)

Available columns in df:
0: Session
1: Title
2: Abstract
3: Abstract ID
4: Technical Community
5: Profile: First Name
6: Profile: Last Name
7: Title and Abstract
8: Session Code

Dataframe contains 1559 rows and 9 columns

Available columns in df_hybrid_presentations:
0: Session
1: Title
2: Abstract
3: Abstract ID
4: Technical Community
5: Presenter: First Name
6: Presenter: Last Name
7: Title and Abstract
8: cluster_id

Dataframe contains 6 rows and 9 columns



In [13]:
# First, define the column mapping lists before calling the function
hybrid_columns_to_map = [
    'Session',
    'Title', 
    'Abstract',
    'Abstract ID',
    'Technical Community', 
    'Presenter: First Name',
    'Presenter: Last Name',
    'Title and Abstract', 
    # Add more columns as needed - copy from print_df_info output above
]

df_columns_to_map = [
    'Session',
    'Title', 
    'Abstract',
    'Abstract ID',
    'Technical Community', 
    'Profile: First Name',
    'Profile: Last Name',
    'Title and Abstract',
    # Add corresponding columns in same order
]

def add_hybrid_presentations_to_df(df, df_hybrid_presentations, df_sessions, session_column_name='Session Code', 
                                  hybrid_columns_to_map=None, df_columns_to_map=None):
    """
    Add hybrid presentations to the main dataframe with proper column mapping and session codes.
    
    Parameters:
    - df: Main presentations dataframe
    - df_hybrid_presentations: Hybrid presentations dataframe
    - df_sessions: Sessions dataframe with cluster_id and hybrid_invited_presentations
    - session_column_name: Name of the session column in df
    - hybrid_columns_to_map: List of column names from df_hybrid_presentations to map (excluding cluster_id)
    - df_columns_to_map: List of corresponding column names in df (same order as hybrid_columns_to_map)
    
    Returns:
    - Tuple: (Combined dataframe with hybrid presentations added, Updated df_sessions)
    """
    
    # Create copies to avoid modifying the originals
    df_combined = df.copy()
    df_sessions_updated = df_sessions.copy()
    
    # Get column mappings (exclude cluster_id as it's not needed)
    df_columns = set(df.columns)
    hybrid_columns = set(df_hybrid_presentations.columns) - {'cluster_id'}  # Exclude cluster_id
    
    # Create mapping for columns
    column_mapping = {}
    
    if hybrid_columns_to_map is not None and df_columns_to_map is not None:
        # Manual mapping provided
        if len(hybrid_columns_to_map) != len(df_columns_to_map):
            raise ValueError("hybrid_columns_to_map and df_columns_to_map must have the same length")
        
        for hybrid_col, df_col in zip(hybrid_columns_to_map, df_columns_to_map):
            if hybrid_col in hybrid_columns and df_col in df_columns:
                column_mapping[hybrid_col] = df_col
            else:
                if hybrid_col not in hybrid_columns:
                    print(f"Warning: '{hybrid_col}' not found in df_hybrid_presentations columns")
                if df_col not in df_columns:
                    print(f"Warning: '{df_col}' not found in df columns")
    else:
        # Automatic mapping (original logic)
        # Map hybrid columns to df columns
        for hybrid_col in hybrid_columns:
            if hybrid_col in df_columns:
                column_mapping[hybrid_col] = hybrid_col
            else:
                # Try to find a matching column in df (case-insensitive or similar)
                for df_col in df_columns:
                    if hybrid_col.lower().replace('_', ' ') == df_col.lower().replace('_', ' '):
                        column_mapping[hybrid_col] = df_col
                        break
    
    print(f"Column mapping: {column_mapping}")
    
    # Track the current max index to know where new indices start
    current_max_index = df_combined.index.max()
    next_index = current_max_index + 1
    
    # Process each session that has hybrid presentations
    for session_idx, session_row in df_sessions_updated.iterrows():
        cluster_id = session_row[session_organizer.COLUMNS['CLUSTER_ID']]
        hybrid_indices = session_row[session_organizer.COLUMNS['HYBRID_INVITED_PRESENTATIONS']]
        
        if len(hybrid_indices) > 0:
            # Get hybrid presentations for this session
            hybrid_pres_for_session = df_hybrid_presentations.loc[hybrid_indices].copy()
            
            # Map columns and add session code
            mapped_hybrid_pres = pd.DataFrame()  # Don't preserve original index
            
            # Map existing columns
            for hybrid_col, df_col in column_mapping.items():
                if hybrid_col in hybrid_pres_for_session.columns:
                    mapped_hybrid_pres[df_col] = hybrid_pres_for_session[hybrid_col].values  # Use .values to avoid index issues
            
            # Add session code (cluster_id from df_sessions)
            mapped_hybrid_pres[session_column_name] = cluster_id
            
            # Add any missing columns from df with NaN values
            for col in df.columns:
                if col not in mapped_hybrid_pres.columns:
                    mapped_hybrid_pres[col] = pd.NA
            
            # Reorder columns to match df
            mapped_hybrid_pres = mapped_hybrid_pres.reindex(columns=df.columns)
            
            # Assign specific indices to the new hybrid presentations
            num_hybrid_pres = len(hybrid_pres_for_session)
            new_indices = list(range(next_index, next_index + num_hybrid_pres))
            mapped_hybrid_pres.index = new_indices
            
            # Append to combined dataframe
            df_combined = pd.concat([df_combined, mapped_hybrid_pres])
            
            # Update df_sessions to include the new indices in gen_presentation_indices
            # FIX: Use .at instead of .loc for setting list values
            current_gen_indices = df_sessions_updated.at[session_idx, session_organizer.COLUMNS['GEN_PRESENTATION_INDICES']]
            updated_gen_indices = current_gen_indices + new_indices
            df_sessions_updated.at[session_idx, session_organizer.COLUMNS['GEN_PRESENTATION_INDICES']] = updated_gen_indices
            
            # Update the session size
            df_sessions_updated.at[session_idx, session_organizer.COLUMNS['SESSION_SIZE']] = len(updated_gen_indices)
            
            print(f"Added {len(hybrid_indices)} hybrid presentations to session {cluster_id}")
            print(f"New indices: {new_indices}")
            
            # Update next_index for the next session
            next_index += num_hybrid_pres
    
    return df_combined, df_sessions_updated

# Usage with manual mapping:
df_with_hybrid, df_sessions_updated = add_hybrid_presentations_to_df(
    df, 
    df_hybrid_presentations, 
    df_sessions, 
    session_column_name,
    hybrid_columns_to_map=hybrid_columns_to_map,
    df_columns_to_map=df_columns_to_map
)
print(f"Original df shape: {df.shape}")
print(f"Combined df shape: {df_with_hybrid.shape}")
print(f"Added {df_with_hybrid.shape[0] - df.shape[0]} hybrid presentations")

# Check for duplicate indices
print(f"Original df index range: {df.index.min()} to {df.index.max()}")
print(f"Combined df index range: {df_with_hybrid.index.min()} to {df_with_hybrid.index.max()}")
print(f"Any duplicate indices? {df_with_hybrid.index.duplicated().any()}")

# Verify session assignments
session_counts = df_with_hybrid[session_column_name].value_counts().sort_index()
print(f"\nSession counts after adding hybrid presentations:")
print(session_counts.head(10))

# Show updated session sizes
print(f"\nUpdated session sizes:")
print(df_sessions_updated[[session_organizer.COLUMNS['CLUSTER_ID'], session_organizer.COLUMNS['SESSION_SIZE']]].head(10))


Column mapping: {'Session': 'Session', 'Title': 'Title', 'Abstract': 'Abstract', 'Abstract ID': 'Abstract ID', 'Technical Community': 'Technical Community', 'Presenter: First Name': 'Profile: First Name', 'Presenter: Last Name': 'Profile: Last Name', 'Title and Abstract': 'Title and Abstract'}
Added 3 hybrid presentations to session 0
New indices: [1559, 1560, 1561]
Added 3 hybrid presentations to session 1
New indices: [1562, 1563, 1564]
Original df shape: (1559, 9)
Combined df shape: (1565, 9)
Added 6 hybrid presentations
Original df index range: 0 to 1558
Combined df index range: 0 to 1564
Any duplicate indices? False

Session counts after adding hybrid presentations:
Session Code
0    10
1    12
2    13
3    16
4    14
5    22
6    16
7    18
8    11
9    14
Name: count, dtype: int64

Updated session sizes:
   cluster_id  session_size
0           0            10
1           1            12
2           2            13
3           3            16
4           4            14
5        

In [14]:
def create_updated_embeddings(df_presentation_embeddings, df_hybrid_embeddings, df_sessions_updated, df_with_hybrid):
    """
    Create updated embeddings dataframe that includes hybrid presentation embeddings
    at their new indices in df_with_hybrid.
    
    Parameters:
    - df_presentation_embeddings: Original presentation embeddings
    - df_hybrid_embeddings: Hybrid presentation embeddings
    - df_sessions_updated: Updated sessions dataframe with new indices
    - df_with_hybrid: Combined dataframe with hybrid presentations
    
    Returns:
    - Updated embeddings dataframe with hybrid embeddings added
    """
    
    # Start with a copy of the original embeddings
    df_embeddings_updated = df_presentation_embeddings.copy()
    
    # Track which hybrid indices have been processed to avoid duplicates
    processed_hybrid_indices = set()
    
    # Process each session that has hybrid presentations
    for session_idx, session_row in df_sessions_updated.iterrows():
        hybrid_indices = session_row[session_organizer.COLUMNS['HYBRID_INVITED_PRESENTATIONS']]
        gen_indices = session_row[session_organizer.COLUMNS['GEN_PRESENTATION_INDICES']]
        
        if len(hybrid_indices) > 0:
            # Find the new indices that were added for hybrid presentations
            # These are the indices in gen_indices that are beyond the original df size
            original_max_index = df_presentation_embeddings.index.max()
            new_hybrid_indices = [idx for idx in gen_indices if idx > original_max_index]
            
            # Map original hybrid indices to new indices
            if len(new_hybrid_indices) == len(hybrid_indices):
                for orig_hybrid_idx, new_idx in zip(hybrid_indices, new_hybrid_indices):
                    if orig_hybrid_idx not in processed_hybrid_indices:
                        # Get the embedding for this hybrid presentation
                        hybrid_embedding_row = df_hybrid_embeddings.loc[orig_hybrid_idx].copy()
                        
                        # Add it to the updated embeddings at the new index
                        df_embeddings_updated.loc[new_idx] = hybrid_embedding_row
                        
                        processed_hybrid_indices.add(orig_hybrid_idx)
                        
                        print(f"Added hybrid embedding: original index {orig_hybrid_idx} -> new index {new_idx}")
    
    # Sort by index to maintain order
    df_embeddings_updated = df_embeddings_updated.sort_index()
    
    print(f"Original embeddings shape: {df_presentation_embeddings.shape}")
    print(f"Updated embeddings shape: {df_embeddings_updated.shape}")
    print(f"Added {df_embeddings_updated.shape[0] - df_presentation_embeddings.shape[0]} hybrid embeddings")
    
    # Verify that embeddings indices match df_with_hybrid indices
    missing_indices = set(df_with_hybrid.index) - set(df_embeddings_updated.index)
    if missing_indices:
        print(f"Warning: Missing embeddings for indices: {sorted(missing_indices)}")
    else:
        print("✓ All presentation indices have corresponding embeddings")
    
    return df_embeddings_updated

# Create the updated embeddings dataframe
df_presentation_embeddings_updated = create_updated_embeddings(
    df_presentation_embeddings, 
    df_hybrid_embeddings, 
    df_sessions_updated, 
    df_with_hybrid
)

# Verify the indices match
print(f"\nIndex verification:")
print(f"df_with_hybrid index range: {df_with_hybrid.index.min()} to {df_with_hybrid.index.max()}")
print(f"df_presentation_embeddings_updated index range: {df_presentation_embeddings_updated.index.min()} to {df_presentation_embeddings_updated.index.max()}")
print(f"Indices match: {set(df_with_hybrid.index) == set(df_presentation_embeddings_updated.index)}")

Added hybrid embedding: original index 0 -> new index 1559
Added hybrid embedding: original index 1 -> new index 1560
Added hybrid embedding: original index 2 -> new index 1561
Added hybrid embedding: original index 3 -> new index 1562
Added hybrid embedding: original index 4 -> new index 1563
Added hybrid embedding: original index 5 -> new index 1564
Original embeddings shape: (1559, 385)
Updated embeddings shape: (1565, 385)
Added 6 hybrid embeddings
✓ All presentation indices have corresponding embeddings

Index verification:
df_with_hybrid index range: 0 to 1564
df_presentation_embeddings_updated index range: 0 to 1564
Indices match: True


### Analyze Sessions

#### Analyze results without invited hybrid presentations

In [15]:
embeddings_only = df_presentation_embeddings.drop(columns=[session_organizer.COLUMNS['EMBEDDING_MODEL']])
pres_similarities_matrix = embedding_model.similarity(embeddings_only.values, embeddings_only.values)
# Convert to numpy if needed
if hasattr(pres_similarities_matrix, 'cpu'):
    pres_similarities_matrix = pres_similarities_matrix.cpu().numpy()
elif hasattr(pres_similarities_matrix, 'numpy'):
    pres_similarities_matrix = pres_similarities_matrix.numpy()

df['presentation_session_fit'],df_sessions['session_coherence'], df_sessions['session_distinctiveness'], df_session_session_similarity  = session_organizer.calculate_placement_metrics(
    df_presentations=df,
    df_sessions=df_sessions,
    pres_similarities_matrix=pres_similarities_matrix,
    session_column_name=session_column_name
)

#### Analyze results with invited hybrid presentations

In [16]:
embeddings_only_updated = df_presentation_embeddings_updated.drop(columns=[session_organizer.COLUMNS['EMBEDDING_MODEL']])
pres_similarities_matrix_updated = embedding_model.similarity(embeddings_only_updated.values, embeddings_only_updated.values)
# Convert to numpy if needed
if hasattr(pres_similarities_matrix_updated, 'cpu'):
    pres_similarities_matrix_updated = pres_similarities_matrix_updated.cpu().numpy()
elif hasattr(pres_similarities_matrix_updated, 'numpy'):
    pres_similarities_matrix_updated = pres_similarities_matrix_updated.numpy()

df_with_hybrid['presentation_session_fit'],df_sessions_updated['session_coherence'], df_sessions_updated['session_distinctiveness'], df_session_session_similarity_updated  = session_organizer.calculate_placement_metrics(
    df_presentations=df_with_hybrid,
    df_sessions=df_sessions_updated,
    pres_similarities_matrix=pres_similarities_matrix_updated,
    session_column_name=session_column_name
)

### Create Session Titles & Keywords

#### Ollama

In [14]:
# Test if Ollama is accessible
import requests
try:
    response = requests.get("http://localhost:11434/api/tags")
    print(f"Ollama status: {response.status_code}")
    if response.status_code == 200:
        models = response.json()['models']
        print(f"Available models: {[m['name'] for m in models]}")
    else:
        print("Ollama server not responding correctly")
except Exception as e:
    print(f"Cannot connect to Ollama: {e}")
    print("Make sure to run 'ollama serve' first")

Ollama status: 200
Available models: ['llama3.2:latest']


In [15]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='ollama:llama3.2:latest')
# Generate titles and keywords for all sessions
# Display sample results
print(df_sessions_sample.head().to_string(index=False))

Using model: llama3.2:latest
Processing session 0 (1/3)...
  ✓ Generated titles for session 0
Processing session 1 (2/3)...
  ✓ Generated titles for session 1
Processing session 2 (3/3)...
  ✓ Generated titles for session 2

Total processing time: 31.62 seconds
Average time per session: 10.54 seconds
 cluster_id  session_size                                                 gen_presentation_indices hybrid_invited_presentations                                                                  final_session_title  session_coherence  session_distinctiveness                                                              Ollama: llama3.2:latest Title 1                                                     Ollama: llama3.2:latest Title 2                                                              Ollama: llama3.2:latest Title 3                                                                                       Ollama: llama3.2:latest Keywords
          0             7                           

#### Sentence Transformers

In [ ]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='llama-3.2-local')
# Generate titles and keywords for all sessions
# df_sessions = generate_session_titles_and_keywords(df_sessions, df, topic_column)

# Display sample results
print(df_sessions_sample.head().to_string(index=False))

In [ ]:
if 'model' in globals() or 'model' in locals():
    del model
    # Optionally, you can try to explicitly trigger garbage collection
    # import gc
    # gc.collect()
    print("LLaMA model has been flagged for unloading. Resources will be freed by the garbage collector.")
else:
    print("Model variable 'model' not found, or already unloaded.")

#### Gemini

In [13]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='gemini-2.0-flash')
# Generate titles and keywords for all sessions
# df_sessions = generate_session_titles_and_keywords(df_sessions, df, topic_column)

# Display sample results
print(df_sessions_sample.head().to_string(index=False))

Processing session 0 (1/3)...
  ✓ Generated titles for session 0
Processing session 1 (2/3)...
  ✓ Generated titles for session 1
Processing session 2 (3/3)...
  ✓ Generated titles for session 2

Total processing time: 6.9847 seconds
Average time per session: 2.3282 seconds
 cluster_id  session_size                                                 gen_presentation_indices hybrid_invited_presentations                                                                  final_session_title  session_coherence  session_distinctiveness                                             Gemini Title 1                                               Gemini Title 2                                        Gemini Title 3                                                                Gemini Keywords
          0             7                                    [777, 937, 980, 1025, 1123, 880, 619]                    [0, 1, 2]                        AI-Powered Remote Sensing for Crop and Soil Health Monitoring   

### Match Committees to Related Sessions

In [14]:
# Read the committee file from CSV/Excel with flexible column selection
committee_file_path = 'ASABE Committees.csv'  # Update this path as needed (can also use .xlsx)

# Load committees
df_committees, committee_name_column, description_column, combined_column = session_organizer.load_committees(
    committee_file_path,
    Committee_Name_column='Committee_Name',  # Actual column name in your file
    Description_column='Description',        # Actual column name in your file
    committee_name_column='Committee_Name',  # Desired output column name
    description_column='Description',        # Desired output column name
    combined_column='Name_Description'       # Combined column for embeddings
)

print(f"Loaded {len(df_committees)} committees")
print(f"Committee name column: {committee_name_column}")
print(f"Description column: {description_column}")
print(f"Combined column: {combined_column}")
print("\nFirst few committees:")
print(df_committees[[committee_name_column, description_column]].head())

# Generate embeddings for committees using the combined column
df_committee_embeddings = session_organizer.embed_documents(df_committees, combined_column, embedding_model)

Loaded 108 committees
Committee name column: Committee_Name
Description column: Description
Combined column: Name_Description

First few committees:
                                      Committee_Name  \
0  ASE-09 Environmental Quality Coordinating Comm...   
1                          ASE-12 Forest Engineering   
2  ASE-134 Fertilizers, Soil Conditioners & US TA...   
3              ASE-16 Engineering for Sustainability   
4  ASE-347 and US TAG TC 347 Data-driven agrifood...   

                                         Description  
0  Leads and coordinates the activities of ASABE ...  
1  Forested landscapes are essential for clean wa...  
2  US Technical Advisory Group for ISO TC 134. Le...  
3  ASE-16 leads and coordinates ASABE activities ...  
4  Standardization in the field of big-picture, d...  


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [15]:
# Find the most similar committees for each session
session_committee_matches = session_organizer.find_most_similar_committees_by_presentations(
    df_sessions, 
    df_presentation_embeddings, 
    df_committees, 
    df_committee_embeddings, 
    top_n=3,
)

Processing session 0 with 7 presentations...
Processing session 1 with 9 presentations...
Processing session 2 with 13 presentations...
Processing session 3 with 16 presentations...
Processing session 4 with 14 presentations...
Processing session 5 with 22 presentations...
Processing session 6 with 16 presentations...
Processing session 7 with 18 presentations...
Processing session 8 with 11 presentations...
Processing session 9 with 14 presentations...
Processing session 10 with 16 presentations...
Processing session 11 with 29 presentations...
Processing session 12 with 12 presentations...
Processing session 13 with 15 presentations...
Processing session 14 with 23 presentations...
Processing session 15 with 17 presentations...
Processing session 16 with 12 presentations...
Processing session 17 with 20 presentations...
Processing session 18 with 16 presentations...
Processing session 19 with 12 presentations...
Processing session 20 with 15 presentations...
Processing session 21 wit

In [16]:
df_sessions = session_organizer.add_committee_matches_to_clusters(df_sessions, session_committee_matches)
# Display a sample of the results
print(f"\nSample of top committee matches:")

print(df_sessions.head(10).to_string(index=False))


Sample of top committee matches:
 cluster_id  session_size                                                                                            gen_presentation_indices hybrid_invited_presentations                                                                  final_session_title  session_coherence  session_distinctiveness                             Top Committee Match                                    2nd Committee Match                                    3rd Committee Match Top Committee Similarity 2nd Committee Similarity 3rd Committee Similarity
          0             7                                                                               [777, 937, 980, 1025, 1123, 880, 619]                    [0, 1, 2]                        AI-Powered Remote Sensing for Crop and Soil Health Monitoring           0.582196                 0.129191            NRES-246 Turf & Landscape Irrigation                         NRES-244 Irrigation Management                          MS-60

## Analyze Manually Edited Session Placements
After session creation, organizers will change placements. This code loads those placements and calculates the similarities of these sessions.

Presentations are placed into sessions based on them having the same session code in the session code column of the presentation spreadsheet. A session dataframe is then created to match the presentation dataframe.

### Check Data File Format

In [5]:
# First, examine the Excel file structure
file_path = "Presentations - Manual Placement.csv"
df_temp = pd.read_csv(file_path)

print("Available columns:")
for i, col in enumerate(df_temp.columns):
    print(f"{i}: {col}")

print(f"\nFile contains {len(df_temp)} rows and {len(df_temp.columns)} columns")
print("\nFirst few rows preview:")
print(df_temp.head())

Available columns:
0: Session
1: Title
2: Abstract
3: Abstract ID
4: Technical Community
5: Profile: First Name
6: Profile: Last Name
7: Title and Abstract
8: Session Code
9: presentation_session_fit

File contains 1559 rows and 10 columns

First few rows preview:
                                             Session  \
0  Thermochemical and Catalytic Conversion of Bio...   
1  Value-Added Chemicals Products and Materials t...   
2  Innovations in Precision Agriculture and Smart...   
3  Conservation Drainage Practices – Current and ...   
4  Water Management and Soil Health under Water S...   

                                               Title  \
0  Catalytic hydrothermal gasification of pinecon...   
1  Progressive Closed-Loop Technologies from Biom...   
2  High Clearance Robotic Irrigation Impacts on S...   
3  Accelerating the adoption of saturated buffers...   
4  Evaluation of Crop Water Use and Productivity ...   

                                            Abstract  Abstrac

In [6]:
# Define your column selections based on the output above
TITLE_COLUMN = 'Title'  # Update based on your file
ABSTRACT_COLUMN = 'Abstract'  # Update based on your file  
ID_COLUMN = 'Abstract ID'  # Update based on your file
SESSION = 'Session Code'  # Update based on your file

### Load Manually Placed Presentations

In [7]:
# Load the data using the session_organizer function
df, title_column, abstract_column, abstract_id_column, topic_column = session_organizer.load_presentations(
    file_path, 
    Title_name=TITLE_COLUMN,
    Abstract_name=ABSTRACT_COLUMN,
    Abstract_ID_name=ID_COLUMN
)

print(f"Loaded {len(df)} presentations successfully")
print(f"Title column: {title_column}")
print(f"Abstract column: {abstract_column}")
print(f"ID column: {abstract_id_column}")
print(f"Topic column: {topic_column}")

Loaded 1559 presentations successfully
Title column: Title
Abstract column: Abstract
ID column: Abstract ID
Topic column: Title and Abstract


### Perform the Embedding

In [8]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', trust_remote_code=True)
print(f"Base model: {embedding_model.model_card_data.base_model}")
df_presentation_embeddings = session_organizer.embed_documents(df, topic_column, embedding_model)
print(f"Embeddings shape: {df_presentation_embeddings.shape}")
print(f"Embedding model used: {df_presentation_embeddings[session_organizer.COLUMNS['EMBEDDING_MODEL']].iloc[0]}")

Base model: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/49 [00:00<?, ?it/s]

Embeddings shape: (1559, 385)
Embedding model used: Unknown (sentence-transformers/all-MiniLM-L6-v2)


### Remove Duplicates and Near-Duplicates

In [9]:
similarity_threshold = 0.99
# Remove near-duplicate presentations based on the similarity threshold
df, df_presentation_embeddings = session_organizer.remove_duplicates(df, df_presentation_embeddings, similarity_func=embedding_model.similarity, threshold=similarity_threshold)


Found 0 near-duplicate presentations to remove (keeping highest index).
Indices to remove: []

Final number of oral presentations: 1559
Final shape of embeddings matrix: (1559, 385)


### Create Sessions

In [ ]:
def create_sessions_from_assignments(df_edited, session_column='Session Code'):
    """Create df_sessions from manually assigned session codes"""

    # Get unique session codes (excluding -1 for unassigned)
    assigned_df = df_edited[df_edited[session_column] != -1]
    
    if assigned_df.empty:
        # No sessions assigned
        df_sessions = pd.DataFrame()
        labels = pd.Series(-1, index=df_edited.index, name=session_column)
        metadata = {
            'n_clusters': 0,
            'n_assigned_items': 0,
            'n_unassigned_items': len(df_edited),
            'n_total_items': len(df_edited),
            'source': 'manually_edited'
        }
        return
    print("Creating sessions from manually assigned session codes...")
    # Group by session code to create clusters
    session_groups = assigned_df.groupby(session_column)
    final_clusters_df_indices = []
    
    for session_code, group in session_groups:
        # Get the DataFrame indices for this session
        cluster_indices = group.index.tolist()
        final_clusters_df_indices.append(cluster_indices)
    
    # Sort clusters by session code for consistency
    final_clusters_df_indices.sort(key=lambda cluster: df_edited.loc[cluster[0], session_column])
    
    # Prepare hybrid data - preserve existing hybrid sessions in their exact locations
    hybrid_cluster_presentations = []
    hybrid_session_titles = []
    
    
    # Ignore hybrid data, use defaults for all clusters
    hybrid_cluster_presentations = [[] for _ in final_clusters_df_indices]
    hybrid_session_titles = [session_organizer.UNSET_SESSION_TITLE_TEXT for _ in final_clusters_df_indices]
    
    # Create output structures using the existing function
    return session_organizer._create_output_structures_with_df_indices(
        final_clusters_df_indices, 
        df_edited, 
        session_column,
        hybrid_cluster_presentations, 
        hybrid_session_titles, 
    )
    

In [ ]:
session_column_name = 'Session Code'
# The create_sessions_from_assignments function returns labels that are -1 for unassigned presentations.
# It also ensures that sessions codes are sequential.
# This is important to maintain consistency with the session codes in df_sessions.
# These labels will match the session codes in df_sessions.
df_sessions, labels, metadata = create_sessions_from_assignments(df)
# Update the original DataFrame with new session label codes. 
df[session_column_name] = labels
print(f"Created {metadata['n_clusters']} sessions with {metadata['n_assigned_items']} presentations.")
print(f"Unassigned Presentations: {metadata['n_unassigned_items']}")

Creating sessions from manually assigned session codes...
Created 101 sessions with 1558 presentations.
Unassigned Presentations: 1


### Analyze Sessions

- session_coherence = "Are presentations within this session similar?" (internal session quality)
- session_distinctiveness = "Is this session's topic unique compared to others?" (relative session positioning)
- presentation_session_fit = "Does this presentation match the topic of others in the session?" (presentation fit)

Session Coherence measures cluster cohesion. It reflects how tighly grouped the topic of presentations within the session are.

Session Distinctiveness measures how unique each session's topic is. High values mean the session has a clear, focused theme that's different from other sessions. Low values suggest either the session mixes different topics or overlaps too much with other sessions.

Presentation-Session Fit is an individual presentations's average similarity to other presentation in its session. Generically, it can be referred to as "within_cluster_fit", "cluster_membership_strength", or "local_cohesion_score".

In [15]:
embeddings_only = df_presentation_embeddings.drop(columns=[session_organizer.COLUMNS['EMBEDDING_MODEL']])
pres_similarities_matrix = embedding_model.similarity(embeddings_only.values, embeddings_only.values)
# Convert to numpy if needed
if hasattr(pres_similarities_matrix, 'cpu'):
    pres_similarities_matrix = pres_similarities_matrix.cpu().numpy()
elif hasattr(pres_similarities_matrix, 'numpy'):
    pres_similarities_matrix = pres_similarities_matrix.numpy()

df['presentation_session_fit'],df_sessions['session_coherence'], df_sessions['session_distinctiveness'], df_session_session_similarity  = session_organizer.calculate_placement_metrics(
    df_presentations=df,
    df_sessions=df_sessions,
    pres_similarities_matrix=pres_similarities_matrix,
    session_column_name=session_column_name
)

### Create Session Titles & Keywords

#### Ollama

In [ ]:
# Test if Ollama is accessible
import requests
try:
    response = requests.get("http://localhost:11434/api/tags")
    print(f"Ollama status: {response.status_code}")
    if response.status_code == 200:
        models = response.json()['models']
        print(f"Available models: {[m['name'] for m in models]}")
    else:
        print("Ollama server not responding correctly")
except Exception as e:
    print(f"Cannot connect to Ollama: {e}")
    print("Make sure to run 'ollama serve' first")

In [ ]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='ollama:llama3.2:latest')
# Generate titles and keywords for all sessions
# Display sample results
print(df_sessions_sample.head().to_string(index=False))

#### Sentence Transformers

In [ ]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='llama-3.2-local')
# Generate titles and keywords for all sessions
# df_sessions = generate_session_titles_and_keywords(df_sessions, df, topic_column)

# Display sample results
print(df_sessions_sample.head().to_string(index=False))

In [ ]:
if 'model' in globals() or 'model' in locals():
    del model
    # Optionally, you can try to explicitly trigger garbage collection
    # import gc
    # gc.collect()
    print("LLaMA model has been flagged for unloading. Resources will be freed by the garbage collector.")
else:
    print("Model variable 'model' not found, or already unloaded.")

#### Gemini

In [ ]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='gemini-2.0-flash')
# Generate titles and keywords for all sessions
# df_sessions = generate_session_titles_and_keywords(df_sessions, df, topic_column)

# Display sample results
print(df_sessions_sample.head().to_string(index=False))

### Match Committees to Related Sessions

In [ ]:
# Read the committee file from CSV/Excel with flexible column selection
committee_file_path = 'ASABE Committees.csv'  # Update this path as needed (can also use .xlsx)

# Load committees
df_committees, committee_name_column, description_column, combined_column = session_organizer.load_committees(
    committee_file_path,
    Committee_Name_column='Committee_Name',  # Actual column name in your file
    Description_column='Description',        # Actual column name in your file
    committee_name_column='Committee_Name',  # Desired output column name
    description_column='Description',        # Desired output column name
    combined_column='Name_Description'       # Combined column for embeddings
)

print(f"Loaded {len(df_committees)} committees")
print(f"Committee name column: {committee_name_column}")
print(f"Description column: {description_column}")
print(f"Combined column: {combined_column}")
print("\nFirst few committees:")
print(df_committees[[committee_name_column, description_column]].head())

# Generate embeddings for committees using the combined column
df_committee_embeddings = session_organizer.embed_documents(df_committees, combined_column, embedding_model)

In [ ]:
# Find the most similar committees for each session
session_committee_matches = session_organizer.find_most_similar_committees_by_presentations(
    df_sessions, 
    df_presentation_embeddings, 
    df_committees, 
    df_committee_embeddings, 
    top_n=3,
)

In [ ]:
df_sessions = session_organizer.add_committee_matches_to_clusters(df_sessions, session_committee_matches)
# Display a sample of the results
print(f"\nSample of top committee matches:")

print(df_sessions.head(10).to_string(index=False))

## Verification
Test to make sure all presentations are included.

In [ ]:
from itertools import chain

all_indices = list(chain.from_iterable(df_sessions[session_organizer.COLUMNS['GEN_PRESENTATION_INDICES']]))
max_index = max(all_indices)
min_index = min(all_indices)
unique_indices = set(all_indices)
expected_indices = set(range(min_index, max_index + 1))
missing_indices = sorted(expected_indices - unique_indices)

print(f"Max index: {max_index}")
print(f"Min index: {min_index}")
print(f"Number of unique indices: {len(unique_indices)}")
print(f"Missing indices: {missing_indices}")
print(f"Any indices skipped? {'Yes' if missing_indices else 'No'}")

# Exporting Data for Web Visualization App

### Include Hybrid Presentations?
Run the cell below to export the results with the hybrid invited presentation included

In [17]:
df  = df_with_hybrid.copy()  # Use the combined DataFrame with hybrid presentations
df_sessions = df_sessions_updated.copy()  # Use the updated sessions DataFrame
df_presentation_embeddings = df_presentation_embeddings_updated.copy()  # Use the updated embeddings DataFrame

## Create sharable dataframe
Abstracts, names and emails should not be posted openly on the web. Remove them from the non-encrypted basic version of the data set.

First, check what columns are available. Then select the ones to remove.

In [19]:
print("Available columns:")
for i, col in enumerate(df.columns):
    print(f"{i}: {col}")

print(f"Dataframe contains {len(df)} rows and {len(df.columns)} columns")


Available columns:
0: Session
1: Title
2: Abstract
3: Abstract ID
4: Technical Community
5: Profile: First Name
6: Profile: Last Name
7: Title and Abstract
8: Session Code
9: presentation_session_fit
Dataframe contains 1565 rows and 10 columns


Copy and paste columns to remove in the list below.

In [20]:
columns_to_drop = [
    'Abstract',
    'Profile: First Name',
    'Profile: Last Name',
    'Title and Abstract',
]
df_no_abstract = df.drop(columns_to_drop, axis=1, errors='ignore')

## Create Encrypted Dataframe

In [21]:
import os
from dotenv import load_dotenv
# Load environment variables
load_dotenv(".env")
if "DATAFRAME_PW" not in os.environ:
    raise ValueError(
        "Dataframe Encryption Password must be in environmental variables. DATAFRAME_PW not found in environment variables. Please set it in your .env file."
    )
else:
    password_df = os.environ["DATAFRAME_PW"]
filename_enc_df = 'encrypted_df.crypt'
# Create an encrypted dataframe for publishing.
import cryptpandas as crp
# Encrypt the DataFrame and save as a pickle
crp.to_encrypted(df, password=password_df, path=filename_enc_df)

In [22]:
# Save the abstract-free DataFrame without encryption
df_no_abstract.to_parquet('df_no_abstract.parquet', compression='snappy')

In [23]:
# Save the Sessions Dataframe
df_sessions.to_parquet('df_sessions.parquet', compression='snappy')

## Save  Similarities
This is both the Presentation-Presentation Similarity Matrix and the Session-Session Similarity Matrix. The web app should not load the embedding model or run the similarity function for speed.

In [ ]:
# Calculate the presentation similarity matrix
embeddings_only = df_presentation_embeddings.drop(columns=[session_organizer.COLUMNS['EMBEDDING_MODEL']])
pres_similarities_matrix = embedding_model.similarity(embeddings_only.values, embeddings_only.values)
# Convert to numpy if needed
if hasattr(pres_similarities_matrix, 'cpu'):
    pres_similarities_matrix = pres_similarities_matrix.cpu().numpy()
elif hasattr(pres_similarities_matrix, 'numpy'):
    pres_similarities_matrix = pres_similarities_matrix.numpy()
pres_similarities_df = pd.DataFrame(pres_similarities_matrix, 
                                    index=df_presentation_embeddings.index, 
                                    columns=df_presentation_embeddings.index)
# Save presentation similarities matrix
pres_similarities_df.to_parquet('pres_similarities_matrix.parquet', compression='snappy')

# Save the session similiarities matrix DataFrame
df_session_session_similarity.to_parquet('session_similarities_matrix.parquet', compression='snappy')

print("✓ Similarity matrices saved as Parquet files")

✓ Similarity matrices saved as Parquet files
